In [1]:

import pandas as pd
movies = pd.read_csv("movies.csv")

In [2]:
def get_movie_genres(movie_id):
    movie = movies[movies["movieId"] == movie_id].iloc[0]
    genres = movie["genres"].split("|")
    return genres

In [3]:
movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
import re

def clean_title(title):
    title = re.sub("[^a-zA-Z0-9 ]", "", title)
    return title

In [5]:

movies["clean_title"] = movies["title"].apply(clean_title)

In [6]:
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    
    return results


In [9]:
movie_id=89745
movie = movies[movies["movieId"] == movie_id]

In [10]:
ratings= pd.read_csv("ratings.csv")

In [11]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [12]:
def get_movie_genres(movie_id):
    movie = movies[movies["movieId"] == movie_id].iloc[0]  # find movie row
    genres = movie["genres"].split("|")  # split genres like Action|Comedy
    return genres


# Collaborative filtering + Genre similarity
def find_similar_movies(movie_id):

    # Find users who rated this movie highly
    similar_users = ratings[
        (ratings["movieId"] == movie_id) & (ratings["rating"] > 4)
    ]["userId"].unique()

    # Movies liked by those users
    similar_user_recs = ratings[
        (ratings["userId"].isin(similar_users)) &
        (ratings["rating"] > 4)
    ]["movieId"]

    # Frequency of recommendations among similar users
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

    # Keep movies liked by at least 10% of similar users
    similar_user_recs = similar_user_recs[similar_user_recs > .10]

    # Compare with ratings from all users
    all_users = ratings[
        (ratings["movieId"].isin(similar_user_recs.index)) &
        (ratings["rating"] > 4)
    ]

    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

    # Create recommendation score
    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]
    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

    # Sort movies by score
    rec_percentages = rec_percentages.sort_values("score", ascending=False)

    # Get top movies and merge with movie details
    recommendations = rec_percentages.head(20).merge(
        movies, left_index=True, right_on="movieId"
    )

    # Get genres of the input movie
    movie_genres = get_movie_genres(movie_id)

    # Calculate how many genres match
    def genre_score(genres):
        g = genres.split("|")
        return len(set(movie_genres).intersection(set(g)))

    recommendations["genre_match"] = recommendations["genres"].apply(genre_score)

    # Combine collaborative score and genre match
    recommendations["final_score"] = (
        recommendations["score"] * 0.7 +
        recommendations["genre_match"] * 0.3
    )

    # Sort by final score
    recommendations = recommendations.sort_values("final_score", ascending=False)

    # Return top 10 movies
    return recommendations[["title", "genres", "score", "genre_match", "final_score"]].head(10)

In [14]:

import ipywidgets as widgets
from IPython.display import display

movie_name_input = widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False
)
recommendation_list = widgets.Output()

def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            results = search(title)
            movie_id = results.iloc[0]["movieId"]
            display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names='value')

display(movie_name_input, recommendation_list)

Text(value='Toy Story', description='Movie Title:')

Output()

In [15]:
from sklearn.metrics import accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt

In [16]:
ratings["liked"] = (ratings["rating"] >= 4).astype(int)

In [17]:
y_true = []
y_scores = []

sample = ratings.sample(500)

In [ ]:
for _, row in sample.iterrows():
    movie_id = row["movieId"]
    
    try:
        recs = find_similar_movies(movie_id)
        score = recs["final_score"].mean()
    except:
        score = 0
    
    y_scores.append(score)
    y_true.append(row["liked"])

In [ ]:
y_pred = [1 if s > 0.5 else 0 for s in y_scores]

accuracy_score(y_true, y_pred)

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label="AUC = %.2f" % roc_auc)
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()
plt.show()